If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec29_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 29: Correlation
v.ekc

Prediction, part two (part one was Lecture 10!): to predict one variable from another we first need to *measure* how tightly they're related. Enter **r**. (Slides: KL Lecture 29 deck.)

**Today's sections:**
1. Prediction recap — heights
2. Association
3. The correlation coefficient
4. What r doesn't tell you

In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

In [ ]:
def r_scatter(r):
    plots.figure(figsize=(5,5))
    "Generate a scatter plot with a correlation approximately r"
    x = np.random.normal(0, 1, 1000)
    z = np.random.normal(0, 1, 1000)
    y = r*x + (np.sqrt(1-r**2))*z
    plots.scatter(x, y, color='darkblue', s=20)
    plots.xlim(-4, 4)
    plots.ylim(-4, 4)

---
## 1. Prediction Recap — Heights

Galton's data and the nearest-neighbor idea from Lecture 10: to predict a child's height, average the children of similar parents.

In [ ]:
# Note: Child heights are the **adult** heights of children in a family
families = Table.read_table('family_heights.csv')
families

In [ ]:
parent_avgs = (families.column('father') + families.column('mother'))/2

In [ ]:
heights = Table().with_columns(
    'Parent Average', parent_avgs,
    'Child', families.column('child'),
)
heights

In [ ]:
heights.scatter('Parent Average', 'Child')

In [ ]:
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2);

In [ ]:
nearby = heights.where('Parent Average', are.between(67.5, 68.5))
nearby_mean = np.average(nearby.column('Child'))
nearby_mean

In [ ]:
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2)
plots.scatter(68, nearby_mean, color='red', s=50);

In [ ]:
def predict_child(h):
    """Predict the height of a child whose parents have a parent average height of p_avg.
    
    The prediction is the average height of the children whose parent average height is
    in the range p_avg plus or minus 0.5.
    """
    nearby = heights.where('Parent Average', are.between(h - 1/2, h + 1/2))
    return np.average(nearby.column('Child'))

In [ ]:
heights_with_predictions = heights.with_columns(
    'Prediction', heights.apply(predict_child, 'Parent Average'))

In [ ]:
heights_with_predictions.scatter('Parent Average')

---
## 2. Association

Hybrid cars: which variables move together? Scatter plots first, always.

In [ ]:
hybrid = Table.read_table('hybrid.csv')
hybrid

In [ ]:
hybrid.sort('msrp', descending=True)

In [ ]:
hybrid.scatter('mpg', 'msrp')

In [ ]:
hybrid.scatter('acceleration', 'msrp')

In [ ]:
suv = hybrid.where('class', 'SUV')
suv.num_rows

In [ ]:
suv.scatter('acceleration', 'msrp')

In [ ]:
suv.scatter('mpg', 'msrp')

Standard units make the association *comparable* — same picture, universal ruler:

In [ ]:
def standard_units(x):
    "Convert any array of numbers to standard units."
    return (x - np.average(x)) / np.std(x)

In [ ]:
Table().with_columns(
    'mpg (standard units)',  standard_units(suv.column('mpg')), 
    'msrp (standard units)', standard_units(suv.column('msrp'))
).scatter(0, 1)
plots.xlim(-3, 3)
plots.ylim(-3, 3);

In [ ]:
suv.scatter('acceleration', 'msrp')

In [ ]:
Table().with_columns(
    'acceleration (standard units)', standard_units(suv.column('acceleration')), 
    'msrp (standard units)',         standard_units(suv.column('msrp'))
).scatter(0, 1)
plots.xlim(-3, 3)
plots.ylim(-3, 3);

---
## 3. The Correlation Coefficient r

**r = the average of the products of standard units.** It measures *linear* association: how consistently do the two variables sit on the same side of their means?

### 📋 Board Reference

| Fact about r | |
|---|---|
| range | always between −1 and 1 |
| r = ±1 | points exactly on a line |
| r = 0 | no *linear* association |
| units | none — it's based on standard units |
| symmetric | r(x, y) = r(y, x) |
| formula | `np.average(su(x) * su(y))` |

In [ ]:
r_scatter(0)

In [ ]:
x = np.arange(1, 7, 1)
y = make_array(2, 3, 1, 5, 2, 7)
t = Table().with_columns(
        'x', x,
        'y', y
    )
t

In [ ]:
t.scatter('x', 'y', s=30, color='red')

In [ ]:
t = t.with_columns(
        'x (standard units)', standard_units(x),
        'y (standard units)', standard_units(y)
    )
t

In [ ]:
t.scatter(2, 3, s=30, color='red')

In [ ]:
t = t.with_columns(
    'product of standard units', t.column(2) * t.column(3))
t

In [ ]:
# r is the average of the products of the standard units

r = np.average(t.column(2) * t.column(3))
r

In [ ]:
def correlation(t, x, y):
    """t is a table; x and y are column labels"""
    x_in_standard_units = standard_units(t.column(x))
    y_in_standard_units = standard_units(t.column(y))
    return np.average(x_in_standard_units * y_in_standard_units)

In [ ]:
correlation(t, 'x', 'y')

In [ ]:
suv.scatter('mpg', 'msrp')

In [ ]:
correlation(suv, 'mpg', 'msrp')

In [ ]:
suv.scatter('acceleration', 'msrp')

In [ ]:
correlation(suv, 'acceleration', 'msrp')

---
## 4. What r Doesn't Tell You ⚠️

Three ways r can mislead (KL deck, slides 14–15):

### Switching Axes

In [ ]:
correlation(t, 'x', 'y')

In [ ]:
t.scatter('x', 'y', s=30, color='red')

In [ ]:
t.scatter('y', 'x', s=30, color='red')

In [ ]:
correlation(t, 'y', 'x')

### Nonlinearity

In [ ]:
new_x = np.arange(-4, 4.1, 0.5)
nonlinear = Table().with_columns(
        'x', new_x,
        'y', new_x**2
    )
nonlinear.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(nonlinear, 'x', 'y')

> **r = 0 does NOT mean "no relationship"** — this parabola has a perfect relationship and zero *linear* correlation.

### Outliers

In [ ]:
line = Table().with_columns(
        'x', make_array(1, 2, 3, 4),
        'y', make_array(1, 2, 3, 4)
    )
line.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(line, 'x', 'y')

In [ ]:
outlier = Table().with_columns(
        'x', make_array(1, 2, 3, 4, 5),
        'y', make_array(1, 2, 3, 4, 0)
    )
outlier.scatter('x', 'y', s=30, color='r')

In [ ]:
correlation(outlier, 'x', 'y')

> **One outlier can wreck r.** Four perfectly-lined-up points plus one stray, and r collapses from 1 to 0. Always look at the scatter, never just the number.

### Ecological Correlations

In [ ]:
sat2014 = Table.read_table('sat2014.csv').sort('State')
sat2014

In [ ]:
sat2014.scatter('Critical Reading', 'Math')

In [ ]:
correlation(sat2014, 'Critical Reading', 'Math')

> **Ecological correlations run high.** Each dot here is a whole *state* — averaging smooths away individual variation, inflating r. A student-level scatter would be much cloudier.

### Try It 1: Eyeball, then compute 😊

1. Guess r for `hybrid` (all cars, not just SUVs): `mpg` vs. `msrp`. Then compute it.

2. Without computing: is r for `msrp` vs. `mpg` (axes switched) different? Check.

In [ ]:
# 1. correlation(hybrid, 'mpg', 'msrp')


In [ ]:
# 2. axes switched


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*About −0.53 — negative (higher mpg, cheaper car) and weaker than the SUV-only −0.67. And switching axes changes nothing: r is symmetric.*

```python
# 1.
correlation(hybrid, 'mpg', 'msrp')      # ≈ -0.53

# 2.
correlation(hybrid, 'msrp', 'mpg')      # identical
```

</details>

---
> **Today's takeaway:** r measures linear association on a universal scale — but it's blind to curves, fragile to outliers, and inflated by aggregation. Look at the scatter. Next: turning r into a prediction *line*.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — correlation
```python
def standard_units(x):
    return (x - np.average(x)) / np.std(x)

def correlation(t, x, y):
    return np.average(standard_units(t.column(x)) * standard_units(t.column(y)))
```